In [2]:
import os
import concurrent.futures
from PIL import Image
from tqdm import tqdm

# --- KONFIGURASI ---
# Gunakan 'r' agar backslash Windows tidak dianggap karakter spesial
BASE_DIR = r"D:\Semester 5\PILA\Final Project UAS\Github1"
QUALITY = 85 
EXTENSIONS = ('.png', '.jpg', '.jpeg')

def convert_single_image(file_info):
    """Fungsi untuk memproses satu gambar"""
    old_path, new_path = file_info
    try:
        with Image.open(old_path) as img:
            # Pastikan folder tujuan ada (jika berbeda)
            img.save(new_path, "WEBP", quality=QUALITY, lossless=False)
        
        # Hapus file asli setelah sukses dikonversi
        os.remove(old_path)
        return True
    except Exception as e:
        return f"Error pada {old_path}: {str(e)}"

# Menggunakan ThreadPoolExecutor agar lebih stabil di Jupyter/Windows
def main():
    tasks = []
    print(f"Sedang memindai folder di: {BASE_DIR}...")
    
    for root, dirs, files in os.walk(BASE_DIR):
        for file in files:
            if file.lower().endswith(EXTENSIONS):
                old_path = os.path.join(root, file)
                new_path = os.path.splitext(old_path)[0] + ".webp"
                tasks.append((old_path, new_path))

    total_files = len(tasks)
    if total_files == 0:
        print("Tidak ditemukan file PNG atau JPG.")
        return

    print(f"Ditemukan {total_files} gambar. Memulai konversi...")

    errors = []
    # ThreadPoolExecutor jauh lebih bersahabat dengan Jupyter Notebook
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
        # Progress bar
        results = list(tqdm(executor.map(convert_single_image, tasks), total=total_files, desc="Converting"))
        
        for res in results:
            if res is not True:
                errors.append(res)

    print("\n--- PROSES SELESAI ---")
    print(f"Berhasil: {total_files - len(errors)}")
    if errors:
        print(f"Gagal: {len(errors)}")
        with open("conversion_errors.log", "w") as f:
            for err in errors:
                f.write(err + "\n")

# Di Notebook, panggil langsung tanpa if __name__ == "__main__": pun tidak apa-apa
# Tapi lebih baik tetap digunakan untuk kebiasaan coding yang baik.
if __name__ == "__main__":
    main()

Sedang memindai folder di: D:\Semester 5\PILA\Final Project UAS\Github1...
Ditemukan 14337 gambar. Memulai konversi...


Converting: 100%|██████████| 14337/14337 [03:01<00:00, 78.81it/s]


--- PROSES SELESAI ---
Berhasil: 14337
